<a href="https://colab.research.google.com/github/aadhavjawahar-sys/Mnist_Prediction_Model/blob/main/Mnist_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [144]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [145]:
# 1. Load MNIST dataset
mnist = tf.keras.datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = np.expand_dims(X_train / 255.0, axis=-1)
X_test = np.expand_dims(X_test / 255.0, axis=-1)

#Added
datagen = ImageDataGenerator(
    rotation_range=15,  # Randomly rotate images in the range (-15 to +15 degrees)
    fill_mode='nearest'  # Fill empty space left by rotation with nearest pixel values
)

# Fit generator on training data
datagen.fit(X_train)


In [146]:
# Defining the network
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.Dense(10, activation="softmax")
])

In [147]:
# Specifying how it will train
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [149]:
#model.fit(X_train,y_train, epochs=5, batch_size=128)
# Train the model with augmented (rotated) batch data
model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=2,
    validation_data=(X_test, y_test)
)

Epoch 1/2
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 176s 93ms/step - accuracy: 0.9726 - loss: 0.0899 - val_accuracy: 0.9847 - val_loss: 0.0422
Epoch 2/2
 411/1875 ━━━━━━━━━━━━━━━━━━━━ 2:06 86ms/step - accuracy: 0.9796 - loss: 0.0676

KeyboardInterrupt: 

In [150]:
from sklearn.metrics import accuracy_score

y_test_pred = np.argmax(model.predict(X_test),axis=1)
y_train_pred = np.argmax(model.predict(X_train),axis=1)

print("Training accuracy: " + str(accuracy_score(y_train, y_train_pred)*100) + "%")
print("Testing accuracy: " + str(accuracy_score(y_test, y_test_pred)*100)+"%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 35s 19ms/step
Training accuracy: 98.83666666666666%
Testing accuracy: 98.59%


In [151]:
import gradio as gr
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def predict_digit(sketch_dict):
    if sketch_dict is None:
        return None, {}

    # Extract composite image from dictionary input
    if isinstance(sketch_dict, dict):
        image_data = sketch_dict.get("composite", None)
    else:
        image_data = sketch_dict

    if image_data is None:
        return None, {}

    # Convert to PIL Image & Grayscale, then resize
    img = Image.fromarray(image_data.astype('uint8')).convert('L')
    img = img.resize((28, 28))

    # Normalize & Invert colors for model prediction
    img_array = np.array(img) / 255.0
    img_array = 1.0 - img_array

    # Reshape for CNN model: (1, 28, 28, 1)
    model_input = np.expand_dims(img_array, axis=(0, -1))

    # Predict probabilities
    predictions = model.predict(model_input, verbose=0)[0]
    top_3_indices = np.argsort(predictions)[-3:][::-1]

    pred_dict = {f"Digit {idx}": float(predictions[idx]) for idx in top_3_indices}

    return img, pred_dict


# --- Launch Gradio Interface ---
with gr.Blocks() as demo:
    gr.Markdown("## ✍️ Digit Recognizer Canvas")

    with gr.Row():
        # Added explicit brush and canvas parameters to ensure canvas renders cleanly
        canvas = gr.Sketchpad(
            label="Draw a single digit (0-9) here",
            type="numpy",
            brush=gr.Brush(colors=["#000000"], default_size=15),  # Explicit black brush
            canvas_size=(280, 280)                                # Fixed canvas size
        )

        processed_img_display = gr.Image(label="Processed 28x28 Input", image_mode="L")
        output_label = gr.Label(num_top_classes=3, label="Top 3 Predictions")

    calc_button = gr.Button("Calculate Prediction", variant="primary")

    calc_button.click(
        fn=predict_digit,
        inputs=canvas,
        outputs=[processed_img_display, output_label]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a74bd5b55b3a805a6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
for i in range(600):
  if (np.argmax(model.predict(X_train[i:i+1]),axis=1) == 9):
    plt.imshow(X_train[i])
    plt.show()